# LiveMIEL-local-neuro

This analytical pipeline is based on the original [LiveMIEL method](https://github.com/cviaai/LiveMIEL). This repository contains modifications to reproduce the results of the article *Novel fluorescent sensor for H3K36me2/3 provides high precision live-cell tracking of epigenetic changes during neuronal differentiation*, featuring an add-on for downloading data from Zenodo alongside custom processing and visualization scripts.

# Getting Started

Follow these steps to set up your local Python environment and install all required dependencies.

### 1. Environment Setup

We recommend using **Conda** to manage an isolated environment with **Python 3.10**:

```bash
# Create a new conda environment
conda create -n livemiel python=3.10 pip -y

# Activate the environment
conda activate livemiel
```

### 2. Installing Dependencies

Install the core image processing, computer vision, and data analysis libraries via `pip`.
```bash
pip install numpy pandas scikit-learn tqdm scikit-image opencv-python Pillow mahotas scipy pingouin scikit-posthocs statannotations matplotlib seaborn plotly
```

If you have the `requirements.txt` file in the same directory, run:
```bash
pip install -r requirements.txt
```

*Note: If you are running this directly inside a Jupyter notebook cell, use the `%pip` command (`%pip install -r requirements.txt`) to ensure it installs into the correct kernel.*


### 3. Installing PyTorch

Depending on your hardware configuration (CPU vs GPU), choose **one** of the following options to install PyTorch:

* **Option A: Standard PyTorch installation via Conda**
  ```bash
  conda install pytorch torchvision torchaudio -c pytorch
  ```

* **Option B: PyTorch with CUDA 12.1 support via Pip (Recommended for NVIDIA GPUs)**
  ```bash
  pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
  ```
  *Note: Make sure to modify the CUDA version (`cu121`) to match your physical GPU drivers.*


In [ ]:
# to check your current torch version
import torch
print(torch.__version__)

### 4. Importing Core LiveMIEL Modules, files and libraries

This step automatically pulls the foundational module files from the original *LiveMIEL* repository. The pipeline preserves the underlying analytical logic while keeping your local workspace modular and unpolluted by manual copy-pasting.

*Note: If you run into network issues, experience GitHub API limits, or simply prefer local workflows, you can manually download `bandpass_segmentation.py`, `classification.py`, `cropping_single_cells.py`, and `features_extraction.py` from the [additional folder](https://github.com/IbragimovAlmaz/LiveMIEL-local-neuro/tree/main/additional) of this repository and place them directly in your working directory.*


In [ ]:
# Automated script to install modules remotely if missing
import os
import urllib.request

BASE_RAW_URL = "https://githubusercontent.com"
modules = ["bandpass_segmentation.py", "classification.py", "cropping_single_cells.py", "features_extraction.py"]

print("Checking for required LiveMIEL local module files...")

for file in modules:
    if not os.path.exists(file):
        try:
            print(f" -> {file} missing. Downloading remote source...", end="")
            urllib.request.urlretrieve(f"{BASE_RAW_URL}/{file}", file)
            print(" [Success]")
        except Exception as e:
            print(f" [Failed]\nConnection error fetching {file}: {e}")
    else:
        # If you downloaded them manually, your console will just show this:
        print(f" -> {file} is verified and present in workspace.")


#### 5. Fetching Experimental Datasets from Zenodo

This step retrieves the required image datasets directly from Zenodo. The pipeline offers two extraction options: an automated inline downloader using `zenodo-get` or a manual download alternative for local offline workflows.

*Note: If you run into network constraints or prefer local execution, you can manually download the archive files from Zenodo using your web browser, extract them, and place them directly in your project data directory. Ensure they are structured in a centralized root directory to match the local path syntax specified in the README.txt files of both this repository and the source Zenodo repository.*


The complete dataset (174 GB raw / 35 GB compressed) supporting this pipeline is hosted on Zenodo. 

* **Zenodo Repository:** DOI: 10.5281/zenodo.20648952

To run the full pipeline, download the required sensor archives and extract them into a centralized root directory to match the local path syntax (e.g., `./IPSC/nuclei_images/AF9-Red_NGN2/0d/` and `./IPSC/single_nuclei_images/AF9-Red_NGN2/0d/`).

*Note: If you want to skip the raw image preprocessing and nucleus extraction phase, you can download only the preprocessed single-nuclei dataset (`single_nuclei_images.zip`), extract it to `./IPSC/single_nuclei_images/`, and comment out/ignore the segmentation part in the main script.*

In [ ]:
# Automated Zenodo Dataset Fetcher Matching README Path Syntax
import os
import zipfile
import urllib.request

# 🔑 FIXED: Set to "IPSC" to perfectly match your Zenodo and GitHub README paths!
DATA_DIR = "IPSC" 
RECORD_ID = "20648952"

SECRET_TOKEN = "eyJhbGciOiJIUzUxMiJ9.eyJpZCI6IjIzNzljOGZjLWI2MGBe..." # Your full token string here

zip_files = [
    "single_nuclei_images.zip", "AF9-Red_NGN2_0-2d.zip", "AF9-Red_NGN2_3-4d.zip",
    "MPP8-Green_NGN2_0-2d.zip", "MPP8-Green_NGN2_3-4d.zip", "PWWP-Red_NGN2_0-2d.zip",
    "PWWP-Red_NGN2_3-4d.zip", 
    #"video_40s.zip", # uncomment if you want additional video dataset
    "ReadMe.txt"]

print(f"Verifying target experimental directory structural boundaries at ./{DATA_DIR}/...")
os.makedirs(DATA_DIR, exist_ok=True)

for file_name in zip_files:
    local_file_path = os.path.join(DATA_DIR, file_name)
    
    if not os.path.exists(local_file_path):
        # 🛠️ URL syntax corrected to prevent network resolution drops
        download_url = f"https://zenodo.org{RECORD_ID}/files/{file_name}?download=1&token={SECRET_TOKEN}"
        try:
            print(f" -> Streaming {file_name} from Zenodo...", end="")
            urllib.request.urlretrieve(download_url, local_file_path)
            print(" [Downloaded]")
            
            if file_name.endswith(".zip"):
                print(f"    -> Unpacking {file_name} structure...", end="")
                with zipfile.ZipFile(local_file_path, 'r') as zip_ref:
                    # Unpacks directly into the IPSC directory
                    zip_ref.extractall(DATA_DIR) 
                print(" [Extracted]")
        except Exception as e:
            print(f" [Failed]\nError acquiring {file_name}: {e}")
    else:
        print(f" -> {file_name} is verified and present in centralized root directory.")

# Alert prompt to guide user manual folder unification steps
print("\n" + "="*80)
print("⚠️ ACTION REQUIRED: MANUAL DIRECTORY UNIFICATION STEPS")
print("="*80)
print(f"Please check the extracted files inside ./{DATA_DIR}/ and arrange them as follows:")
print("1. Combine the days of 'AF9-Red_NGN2_0-2d' and 'AF9-Red_NGN2_3-4d' files.")
print("2. Combine the days of 'MPP8-Green_NGN2_0-2d' and 'MPP8-Green_NGN2_3-4d' files.")
print("3. Combine the days of 'PWWP-Red_NGN2_0-2d' and 'PWWP-Red_NGN2_3-4d' files.")
print("All subfolders (0d to 4d) must reside in their respective combined parent target directories.")
print("="*80)

*Note: After downloading `PWWP-Red_NGN2_0-2d.zip` and `PWWP-Red_NGN2_3-4d.zip`, please **manually combine their contents** into a single folder named `./IPSC/nuclei_images/PWWP-Red_NGN2/` so the subfolders (`0d` through `4d`) sit together. Do it for all sensors as required*


Importing the key libraries:

In [ ]:
# importing libraries

import numpy as np
import os
from skimage import io
FOLDERNAME = 'C:/Users/User/Desktop/LiveMIEL/' # Your local foldername with required files:
                            #bandpass_segmentation.py
                            #cropping_single_cells.py
                            #features_extraction.py
                            #classification.py
import sys

sys.path.append('C:/Users/User/Desktop/LiveMIEL/{}'.format(FOLDERNAME)) # Your local foldername
import bandpass_segmentation as bs
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import ListedColormap
from sklearn import metrics

import classification
from features_extraction import HaralickFeatures, TAS, \
ZernikeMoments, extract, Preprocess, CenterMass, ChromatinFeatures
from sklearn.model_selection import train_test_split, cross_val_score

In [ ]:
# to check your current location if needed
pwd

# Nuclei segmentation

Enter the directory path of the nuclei images for trial segmentation:

In [ ]:
# choose your main directory here
dir = './IPSC/nuclei_images/AF9-Red_NGN2/2d/'

In [ ]:
lst = os.listdir((dir))
lst.sort()

save_dir_nuclei = path = dir.replace('nuclei_images', 'single_nuclei_images', 1)

try:
    os.makedirs(path, exist_ok = True)
    print("Save_dir_nuclei created successfully")
except OSError as error:
    print("Save_dir_nuclei can not be created")

save_dir_info = path = dir.replace('nuclei_images', 'saved_info_single_cells', 1)

try:
    os.makedirs(path, exist_ok = True)
    print("Save_dir_nuclei created successfully")
except OSError as error:
    print("Save_dir_nuclei can not be created")

## Tuning parameters for image segmentation

Choose image for segmentation parameters tuning:

In [ ]:
# Enter image name to tunining segmentation parameters on
image_name = str('Exp_96.TIF')

### Tuning Parameters for Image Segmentation

To achieve optimal segmentation performance on current microscopy dataset, you can tune the core parameters in the execution line:

```python
image_segm = bs.main(image, lowSigm=10, highSigm=70, coeff=1, thresh=0.003, FalsePositBrightness_k=2)
```

#### Parameter Breakdown & Optimization Tips:

| Parameter | Recommended Range | Primary Function & Biological Context |
| :--- | :--- | :--- |
| `lowSigm` | `5` – `12` *(Up to 40)* | **Noise Reduction & Smoothing.** Higher values prevent a single overlapping or irregular nucleus from being falsely split into multiple distinct objects. *Note: Must always be strictly lower than `highSigm`.* |
| `highSigm` | `30` – `70` | **Background Removal.** Controls the upper limit of the gaussian filter. Setting `highSigm` lower (e.g., around `40`) significantly speeds up the code execution. |
| `thresh` | `0.003` – `0.05` | **Intensity Threshold.** Eliminates background image areas where the fluorescence intensity falls below the specified absolute value. |
| `FalsePositBrightness_k` | `1.2` – `2.5` | **False Positive Filtering.** Post-segmentation coefficient to discard artifacts and dim/dying nuclei with low fluorescence intensity from the final dataset. |
| `coeff` | `1` | **Scaling Coefficient.** Internal multiplier for spatial adjustment (default is set to `1`). |

*Quick Validation Presets:* 
* For fast initial testing and validation, the combinations `lowSigm=15 / highSigm=40` or `lowSigm=10 / highSigm=70` generally yield the most stable baseline results.


### Bioimage Segmentation Parameters Reference

The table below outlines our applied parameters during the cell segmentation phase. Settings are optimized individually for each sensor across the neuronal differentiation timeline to account for changing morphology and nuclear size.

| Target Sensor | Differentiation Stage (Day) | lowSigm : highSigm | Notes / Cleaning Filter |
| :--- | :--- | :--- | :--- |
| **AF9-Red** | Day 0 | 70 : 120 | Large nuclear size with high feature density |
| | Day 1 | 65 : 110 | |
| | Day 2 | 45 : 90 | Intermediate nuclear size |
| | Day 3 | 25 : 85 | |
| | Day 4 | 15 : 75 | Small nuclear size; rapid processing |
| **MPP8-Green**| Day 0—2 | 65 : 115 | Adjust if you experience over-segmentation (fragmentation) |
| | Day 3 | 45 : 100 | |
| | Day 4 | 40 : 90 | Faster processing with smoother segmentation boundaries |
| **PWWP-Red** | Day 0—2 | 65 : 115 | Large nuclear size |
| | Day 3—4 | 40 : 85 | Post-differentiation with reduced nuclear size |


### Key Segmentation Heuristic
As a general rule of thumb for this pipeline, adjust your parameters across the differentiation timeline based on changing cell morphology:
* **Initial Days (Day 0–2):** Use wider windows (e.g., `70:115`) to handle large nuclear sizes and mitigate over-segmentation (fragmentation).
* **Intermediate Days (Day 2–3):** Drop down to moderate scales (e.g., `40:95`) as nuclei begin to condense.
* **Final Day (Day 4):** Use tight, low-range windows (e.g., `35:85`) to capture small nuclear profiles with fewer localized features, resulting in much faster compute times.


In [ ]:
print(os.path.join(dir, image_name))
image = io.imread(os.path.join(dir, image_name))
image_segm = bs.main(image, lowSigm=40, highSigm=70, coeff=1, thresh=0.003, FalsePositBrightness_k = 2) #tune parameters here for better segmentation performance

jet = cm.get_cmap('jet', 256)
newcolors = jet(np.linspace(0, 1, np.max(image_segm)))
black = np.array([0, 0, 0, 1])
newcolors[:1, :] = black
newcmp = ListedColormap(newcolors)

fig, ax = plt.subplots(1,2, figsize=(50, 20))
ax[0].axis('off')
ax[0].imshow(image, cmap='gray')
ax[0].set_title('Original image (in RGB)', fontsize=40)
ax[1].axis('Off')
ax[1].imshow(image_segm, cmap=newcmp)
ax[1].set_title('Segmented and labeled image', fontsize=40)
fig.subplots_adjust(wspace=0.02, hspace=0,
                                top=0.9, bottom=0.05, left=0, right=0.75)

## Segmentation + cropping images into single cells

In [ ]:
import cropping_single_cells as csc

raw_img_names = np.empty(0)
number_segmented = np.empty(0, dtype=int)
labels = np.empty(0, dtype=int)
coordinates = np.empty((0, 4), dtype=int)

for i, img in enumerate(lst):
    # print(os.path.join(dir, img))
    image = io.imread(os.path.join(dir, img))
    image_segm = bs.main(image, lowSigm=10, highSigm=70, coeff=1, thresh=0.03, FalsePositBrightness_k = 2) #put tuned parameters for segmentation

    crops, num_segm, l, coord = csc.cropping_cells(image_segm, image, save_dir_nuclei, img, save=True)

    raw_img_names = np.append(raw_img_names, img)
    number_segmented = np.append(number_segmented, num_segm)
    labels = np.append(labels, l)
    coordinates = np.append(coordinates, coord, axis=0)

    np.save(os.path.join(save_dir_info, 'raw_img_names.npy'), raw_img_names)
    np.save(os.path.join(save_dir_info, 'number_segmented.npy'), number_segmented)
    np.save(os.path.join(save_dir_info, 'labels.npy'), labels)
    np.save(os.path.join(save_dir_info, 'coordinates.npy'), coordinates)

# Collecting features from the segmented images

## Features extraction

Path to cropped single-nuclei dataset:

In [ ]:
# Using all days of differentiation for the AF9-Red sensor as an example

directories = ['./IPSC/single_nuclei_images/AF9-Red_NGN2/0d/',
               './IPSC/single_nuclei_images/AF9-Red_NGN2/1d/',
                './IPSC/single_nuclei_images/AF9-Red_NGN2/2d/',
                './IPSC/single_nuclei_images/AF9-Red_NGN2/3d/',
               './IPSC/single_nuclei_images/AF9-Red_NGN2/4d/']

Selected features to use in classification:
  

*   Haralick features

*   TAS features
*   Zernike moments - if **None**, then radius is calculated automatically
*   Center of mass of each nucleus
*   Chromatin features - minimum area of a chromatin "dot", maximum area of a chromatin "dot", mean area of a chromatin "dot", total area of all chromatin "dots"
*   feat_num - number of each selected features
*   objects_ - number of the nuclei to classify in each class
*   object_class - names of the cell families








In [ ]:
features = {'features': [HaralickFeatures(), TAS(), ZernikeMoments(None), CenterMass(), ChromatinFeatures()],
            'feat_num' : [13, 54, 25, 2, 4]}

for i in range(len(directories)):
    obj_key = 'objects_' + str(i)
    obj_value = len(os.listdir(directories[i]))
    class_key = 'object_' + str(i) + '_class'
    class_value = str(input("Object class for \n%s \n" %directories[i]))
    features[obj_key] = obj_value
    features[class_key] = class_value

preproc_commands = ['meanvar', 'pad'] # can be ['meanvar', 'pad'] or one of them, or empty

In [ ]:
df, X, y = \
classification.dataCollection(directories, features, preproc_commands, normalize=True, enhance=True)
# if normalize=True -- images are normalized to 0-255 prior to fetures extraction
# if enhance=True -- images' brightnesses are normalized to common brightness=0.09

## Features averaging

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_std = scaler.fit_transform(X)

y_arr = np.array(list(y))

print(y_arr, y_arr.shape)

Features averaged over all nuclei within original images

In [ ]:
import re

num_sorted = np.empty(0, dtype=int)
for i in range(len(directories)):
    lst = os.listdir((directories[i]))
    lst.sort()

    num_segm = np.empty(0, dtype=int)

    ref_img = re.search('(.*)_', lst[0]).group() #extraction of original microscopic image name from single nuclei image name
    counter = 0

    for img in lst:
        cur_raw_img = re.search('(.*)_', img).group() #extraction of original microscopic image name from single nuclei image name
        print(ref_img, cur_raw_img)
        if cur_raw_img == ref_img:
            counter += 1
        if cur_raw_img > ref_img:
            num_segm = np.append(num_segm, counter)
            ref_img = cur_raw_img
            counter = 1
    num_segm = np.append(num_segm, counter)
    num_sorted = np.append(num_sorted, num_segm)

In [ ]:
features_num = sum(features['feat_num'])

center_vector = []
y_names = np.empty(0, dtype = np.dtype('U25'))

X_y = np.append(X_std, y_arr.reshape(len(y_arr), 1), axis = 1)
X_y = X_y[X_y[:, features_num].argsort()]

start_r = 0
stop_r = 0

for i in range(len(num_sorted)):
  stop_r += num_sorted[i]
  center_vector.append((np.mean(np.asarray((X_y[start_r : stop_r, :features_num]), dtype='float64', order='C'), axis = 0)))
  y_names = np.append(y_names, X_y[start_r][features_num])
  start_r = stop_r

X = center_vector
y = y_names

Features averaged over N nuclei within image class

In [ ]:
N = 60 # averaging over 60 nuclei was considered to be basic
#N_set = [60, 60, 60, 60, 60] - N value for each nuclei class

In [ ]:
features_num = np.sum(features['feat_num'])

center_vector = []
y_names = np.empty(0, dtype = np.dtype('U25'))

X_y = np.append(X_std, y_arr.reshape(len(y_arr), 1), axis = 1)
X_y = X_y[X_y[:, features_num].argsort()]

start_r = 0
stop_r = 0

for i in range(len(directories)):
  #N = N_set[i]
  for j in range(len(os.listdir(directories[i]))//N):
    stop_r += N
    center_vector.append((np.mean(np.asarray((X_y[start_r : stop_r, :features_num]), dtype='float64', order='C'), axis = 0)))
    y_names = np.append(y_names, X_y[start_r][features_num])
    start_r = stop_r

  if len(os.listdir(directories[i]))/N == len(os.listdir(directories[i]))//N:
    continue
  else:
    stop_r += len(os.listdir(directories[i])) % N
    center_vector.append((np.mean(np.asarray((X_y[start_r : stop_r, :features_num]), dtype='float64', order='C'), axis = 0)))
    y_names = np.append(y_names, X_y[start_r][features_num])
    start_r = stop_r

X = center_vector
y = y_names

## Features preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler1 = StandardScaler()

In [ ]:
X_std = scaler1.fit_transform(X)

print(np.sum(np.double(np.isnan(X))) / np.prod(np.shape(X)))
print(X_std)

# PCA

## Dataset preprocessing

In [ ]:
labels = np.reshape(y,(y.shape[0],1))
final_data = np.concatenate([X_std, labels],axis=1)

In [ ]:
feature_names = np.empty(0)

for i in range(13):
    feature_names = np.append(feature_names, 'Haralick features ' + str(i+1))

for i in range(13, 67):
    feature_names = np.append(feature_names, 'TAS ' + str(i-12))

for i in range(67, 92):
    feature_names = np.append(feature_names, 'Zernike moments ' + str(i-66))

feature_names = np.append(feature_names, 'Center of mass (x)')
feature_names = np.append(feature_names, 'Center of mass (y)')
feature_names = np.append(feature_names, 'Chromatin features (min)')
feature_names = np.append(feature_names, 'Chromatin features (max)')
feature_names = np.append(feature_names, 'Chromatin features (mean)')
feature_names = np.append(feature_names, 'Chromatin features (total)')

In [ ]:
import pandas as pd

cur_dataset = pd.DataFrame(final_data)
features_classes = feature_names
features_labels = np.append(features_classes,'label')
cur_dataset.columns = features_labels

## Explained variance for PCA

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components = 10)

X_train_pca = pca.fit_transform(X_std)

exp_var_pca = pca.explained_variance_ratio_

cum_sum_eigenvalues = np.cumsum(exp_var_pca)

for i in range(3):
    print(
        "For PC",
        i+1,
        "the explained variance is :",
        exp_var_pca[i],
    )

plt.bar(range(0,len(exp_var_pca)), exp_var_pca, alpha=0.5, align='center', label='Individual explained variance')
plt.step(range(0,len(cum_sum_eigenvalues)), cum_sum_eigenvalues, where='mid',label='Cumulative explained variance')
plt.ylabel('Explained variance ratio')
plt.xlabel('Principal component index')
plt.legend(loc='center right')
plt.tight_layout()
plt.show()

# PCA plots

### Interactive 3D PCA plot

In [ ]:
import plotly.express as px

from sklearn.decomposition import PCA
pca_data = PCA(n_components = 3)
principalComponents = pca_data.fit_transform(X_std)

principal_Df = pd.DataFrame(data = principalComponents
             , columns = ['principal component 1', 'principal component 2', 'principal component 3'])

import seaborn as sns

fig_3d = px.scatter_3d(
    principalComponents, x=0, y=1, z=2,
    color=y_names, labels={'color': 'days'})

fig_3d.update_traces(marker_size=5)

fig_3d.show()

# Choose the file name here:
fig_3d.write_html("AF9-Red.html")

### 2D PCA plot

In [ ]:
from sklearn.decomposition import PCA
pca_data = PCA(n_components = 3)
principalComponents = pca_data.fit_transform(X_std)

principal_Df = pd.DataFrame(data = principalComponents
             , columns = ['principal component 1', 'principal component 2', 'principal component 3'])

plt.figure()
plt.figure(figsize=(10,10))
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlabel('Principal Component - 1',fontsize=20)
plt.ylabel('Principal Component - 2',fontsize=20)
plt.title("PCA of IPSC Dataset - 0-4 days \n (feature values averaged over 60 nuclei)",fontsize=20)
targets = ['Day 0', 'Day 1', 'Day 2', 'Day 3', 'Day 4']
colors = ['#6F518F', '#549EAC', '#B8BC3D', '#D0843E', '#AD2C24']

for target, color in zip(targets,colors):
    indicesToKeep = cur_dataset['label'] == target
    plt.scatter(principal_Df.loc[indicesToKeep, 'principal component 1'],
                principal_Df.loc[indicesToKeep, 'principal component 2'], c = color, s = 80, edgecolors='k')

plt.legend(targets,prop={'size': 15})

for target, color in zip(targets,colors):
    indicesToKeep = cur_dataset['label'] == target
    plt.scatter(np.mean(principal_Df.loc[indicesToKeep, 'principal component 1']),
                np.mean(principal_Df.loc[indicesToKeep, 'principal component 2']), s=350, marker="D", c=color,
                lw=2.5, edgecolors="black")

# Clustering

## Hopkins statistics

In [ ]:
X = principal_Df[['principal component 1', 'principal component 2']]
X = X.dropna(axis=0)

In [ ]:
from sklearn.neighbors import NearestNeighbors
from random import sample
from numpy.random import uniform
from math import isnan
def hopkins(X):
  d = X.shape[1]
  n = len(X)
  m = int(0.1 * n)
  nbrs = NearestNeighbors(n_neighbors=1).fit(X.values)

  rand_X = sample(range(0, n, 1), m)

  ujd = []
  wjd = []
  for j in range(0, m):
     u_dist, _ = nbrs.kneighbors(uniform(np.amin(X,axis=0),np.amax(X,axis=0),d).reshape(1, -1), 2, return_distance=True)
     ujd.append(u_dist[0][1])
     w_dist, _ = nbrs.kneighbors(X.iloc[rand_X[j]].values.reshape(1, -1), 2, return_distance=True)
     wjd.append(w_dist[0][1])

  H = sum(wjd) / (sum(ujd) + sum(wjd))
  if isnan(H):
     print(ujd, wjd)
     H = 0

  return H

Hopkins statistics (H):

*   highly clustered data - H ~ 0
*   clustered data - H < 0.5  
*   random data - H ~ 0.5
*   regular / uniform data - H ~ 1.

In [ ]:
hopkins(X)

### Сomparing Hopkins statistics between sensors (optional)

In [ ]:
# An example for AF9-Red sensor:
hopkins_AF9Red = np.array([])
for i in range(100):
    new_hopkins = np.array([hopkins(X)])
    hopkins_AF9Red = np.concatenate((hopkins_AF9Red, new_hopkins), axis=0)
print(hopkins_AF9Red, hopkins_AF9Red.shape)

# To save obtained dataset of 100-times calculated Hopkins statistics for the AF9-Red sensor
np.save('AF9-Red-hopkins-data.npy', hopkins_AF9Red)

a = np.load('AF9-Red-hopkins-data.npy')
average_af9 = np.mean(a)
print(average_af9)
std_af9 = np.std(a)
print(std_af9)

# This node must be performed for all 3 sensors for the further comparison

Shapiro-Wilk test (normality)

In [ ]:
import pingouin as pg
import scipy.stats as stats

# 0. Data import
af9_npy = np.load('AF9-Red-hopkins-data.npy')
pwwp_npy = np.load('PWWP-Red-hopkins-data.npy')
mpp8_npy = np.load('MPP8-Green-hopkins-data.npy')

# 1. Checking the data distribution
print("--- 1. Shapiro-Wilk test (normality) ---")
# p > 0.05: assuming normal distribution
print(f"AF9-Red:    p-value = {stats.shapiro(af9_npy)[1]:.5f}")
print(f"PWWP-Red:   p-value = {stats.shapiro(pwwp_npy)[1]:.5f}")
print(f"MPP8-Green: p-value = {stats.shapiro(mpp8_npy)[1]:.5f}")

Q-Q plots can visually show the distribution:

In [ ]:
# Making dictionary for more convenient output
datasets = {"AF9-Red": af9_npy, "PWWP-Red": pwwp_npy, "MPP8-Green": mpp8_npy}

# 2. Setting up the grid (1 row, 3 columns)
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

# 3. Making Q-Q plot for each group
for ax, (name, data) in zip(axes, datasets.items()):
    # stats.probplot calculates quantiles and builds a plot
    # plot=ax specifies which plot to draw on
    stats.probplot(data, dist="norm", plot=ax)

    # Customizing the appearance
    ax.set_title(f"Q-Q Plot: {name}")
    ax.set_xlabel("Theoretical quantiles")
    ax.set_ylabel("Sample quantiles")
    ax.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()

# plt.savefig('qq_plots.png', dpi=300)
plt.show()

### Statistical Pipeline for Multi-Group Analysis

This pipeline automatically tests statistical assumptions and chooses between parametric or non-parametric test paths.

#### **Step 1: Data Preparation & Normality Testing**
* **Data Consolidation**: Merges target protein groups (`AF9-Red`, `PWWP-Red`, `MPP8-Green`) into a unified DataFrame.
* **Normality Check**: Runs the Shapiro-Wilk test on each experimental group.

#### **Step 2: Conditional Analysis Path Selection**

* **Path A: Parametric Testing** (If all groups are normally distributed, $p > 0.05$)
    1. **Levene’s Test**: Verifies the homogeneity of variances.
    2. **One-Way ANOVA**: Evaluates global variance across groups and reports **Partial $\eta^2$ (Partial Eta-Squared)** (`np2`) for global effect size.
    3. **Tukey’s HSD Post-hoc**: Computes pairwise comparisons and outputs **Hedges' $g$ / Cohen's $d$** for standardized pairwise effect sizes.

* **Path B: Non-Parametric Testing** (If any group violates normality, $p \le 0.05$)
    1. **Kruskal-Wallis Test**: Evaluates global variance rank differences across groups.
    2. **Mann-Whitney U Post-hoc**: Performs pairwise rank-sum tests using **Bonferroni correction** ($p\text{-corr}$) to control the family-wise error rate.
    3. **Effect Size Estimation**: 
        * Computes **Hedges' $g$** via standard pairwise matrix tests.
        * Computes **Cliff's Delta ($\delta$)** derived from the Common Language Effect Size (CLES) to describe non-parametric effect magnitude.

---

### Effect Size Reference Scale

To help interpret the magnitude of the experimental differences found in the steps above, these standard statistical thresholds can be applied:

| Magnitude | **Hedges' $g$ / Cohen's $d$** (Parametric) | **Cliff's Delta ($\delta$)** (Non-Parametric) |
| :--- | :--- | :--- |
| **Negligible** | $|g| < 0.20$ | $|\delta| < 0.147$ |
| **Small** | $0.20 \le |g| < 0.50$ | $0.147 \le |\delta| < 0.330$ |
| **Medium** | $0.50 \le |g| < 0.80$ | $0.330 \le |\delta| < 0.474$ |
| **Large** | $|g| \ge 0.80$ | $|\delta| \ge 0.474$ |


In [ ]:
# 4. Assemble unified dataframe
values = np.concatenate([af9_npy, pwwp_npy, mpp8_npy])
groups = (['AF9-Red'] * len(af9_npy) + 
          ['PWWP-Red'] * len(pwwp_npy) + 
          ['MPP8-Green'] * len(mpp8_npy))
df_stats = pd.DataFrame({'value': values, 'group': groups})

# 5. Check group normality
shapiro_p_values = np.array([
    stats.shapiro(af9_npy)[1], 
    stats.shapiro(pwwp_npy)[1], 
    stats.shapiro(mpp8_npy)[1]
])

# 6. Statistical analysis
if np.all(shapiro_p_values > 0.05):
    print("p > 0.05: Normal distributions confirmed. Proceeding with Parametric Path...")
    print("\n--- Levene's Test (Homogeneity of Variances) ---")
    levene_p = stats.levene(af9_npy, pwwp_npy, mpp8_npy)[1]
    print(f"Levene's test p-value = {levene_p:.5f}")
    
    print("\n--- One-Way ANOVA ---")
    anova_table = pg.anova(data=df_stats, dv='value', between='group', detailed=True)
    print(anova_table[['Source', 'SS', 'DF', 'MS', 'F', 'p-unc', 'np2']]) 

    print("\n--- Post-hoc Pairwise Tukey (with Cohen's d / Hedges g) ---")
    tukey_table = pg.pairwise_tukey(data=df_stats, dv='value', between='group')
    print(tukey_table[['A', 'B', 'diff', 'se', 'T', 'p-tukey', 'hedges']])  
else:
    print("Normality assumption violated. Proceeding with Non-Parametric Path...")
    print("\n--- Global Kruskal–Wallis Test ---")
    kruskal_table = pg.kruskal(data=df_stats, dv='value', between='group')
    print(kruskal_table)

    print("\n--- Pairwise U-test with Bonferroni Correction ---")
    pairwise_nonparam = pg.pairwise_tests(data=df_stats, dv='value', between='group', 
                                      parametric=False, padjust='bonf')
    print(pairwise_nonparam[['A', 'B', 'U-val', 'p-unc', 'p-corr', 'hedges']]) 
    
    print("\n--- Pairwise U-test with Bonferroni Correction + Cliff's Delta ---")
    pairwise_nonparam = pg.pairwise_tests(data=df_stats, dv='value', between='group', 
                                          parametric=False, padjust='bonf', effsize='cles')
    pairwise_nonparam['cliffs_delta'] = 2 * pairwise_nonparam['cles'] - 1
    print(pairwise_nonparam[['A', 'B', 'U-val', 'p-unc', 'p-corr', 'cliffs_delta']]) 

Plotting histogram:

In [ ]:
import seaborn as sns
import plotly.express as px
import matplotlib as mtl
from pylab import rcParams

mtl.rcParams['legend.frameon'] = 'False' # can be changed to'True'
rcParams['figure.figsize'] = 7, 7
rcParams['font.size'] = 14

af9_npy = np.load('AF9-Red-hopkins-data.npy')
pwwp_npy = np.load('PWWP-Red-hopkins-data.npy')
mpp8_npy = np.load('MPP8-Green-hopkins-data.npy')

average_af9 = np.mean(af9_npy)
average_pwwp = np.mean(pwwp_npy)
average_mpp8 = np.mean(mpp8_npy)

std_af9 = np.std(af9_npy)
std_pwwp = np.std(pwwp_npy)
std_mpp8 = np.std(mpp8_npy)

data = np.array([[average_af9, std_af9],
[average_pwwp, std_pwwp],
[average_mpp8, std_mpp8]])

fig, ax = plt.subplots()

labelsx = ['AF9-Red', 'PWWP-Red', 'MPP8-Green']
x = np.arange(len(labelsx))  # the label locations
plt.xticks(x, labelsx)
# tilt of the x labels
plt.xticks(fontsize=20, weight='bold', fontname='arial', rotation=10)
plt.yticks(fontsize=20, weight='bold', fontname='arial')

width = 0.6  # the width of the bars

colors = ['black', 'darkgray', 'dimgray']
labels_hist = ['AF9-Red','PWWP-Red','MPP8-Green']
hatches = [None, None, None]

# Drawing a graph according to custom style
rects1 = ax.bar(x, data[:,0], width, color=colors[:], edgecolor=None, label=labels_hist[:], linewidth=1.5, hatch = hatches[:])

# --- Customization of the first column only ----
rects1[0].set_facecolor('white')       # Fills the first column with white
rects1[0].set_edgecolor('black')       # Adds a black outline to it
rects1[0].set_linewidth(4)             # Sets a large outline thickness (can be changed)
# -----------------------------------------------
    
plt.title('Hopkins statistics for clustered data quality', fontsize=20, weight='bold', fontname='arial', pad=40)

plt.ylabel('Hopkins statistics (H)', fontsize=20, fontname='arial', weight='bold')

plt.ylim(0, 0.5)

for i in range(len(labelsx)):
    plt.errorbar( i, data[i,0], yerr = data[i,1], ecolor='black', capsize=10, elinewidth=5, markeredgewidth=5)

# statistical annotation
def draw_brace(ax, xspan, yy, text):
    """Draws an annotated brace on the axes."""
    xmin, xmax = xspan
    xspan = xmax - xmin
    ax_xmin, ax_xmax = ax.get_xlim()
    xax_span = ax_xmax - ax_xmin

    ymin, ymax = ax.get_ylim()
    yspan = ymax - ymin
    resolution = int(xspan/xax_span*100)*2+1 # guaranteed uneven
    beta = 100./xax_span # the higher this is, the smaller the radius

    x = np.linspace(xmin, xmax, resolution)
    x_half = x[:int(resolution/2)+1]
    y_half_brace = (1/(1.+np.exp(-beta*(x_half-x_half[0])))
                    + 1/(1.+np.exp(-beta*(x_half-x_half[-1]))))
    y = np.concatenate((y_half_brace, y_half_brace[-2::-1]))
    y = yy + (.05*y - .01)*yspan # adjust vertical position

    ax.autoscale(False)
    ax.plot(x, y, color='white', lw=1)

    ax.text((xmax+xmin)/2., yy+.07*yspan, text, ha='center', va='bottom', size=36, fontname='arial')

import matplotlib.patches as mpatches

ax.annotate("",
            xy=(0, 0.43), xycoords='data',
            xytext=(2, 0.43), textcoords='data',
            arrowprops=dict(arrowstyle="-", connectionstyle="arc3", linewidth=4))
ax.annotate("",
            xy=(0, 0.38), xycoords='data',
            xytext=(1, 0.38), textcoords='data',
            arrowprops=dict(arrowstyle="-", connectionstyle="arc3", linewidth=4))
ax.annotate("",
            xy=(1, 0.33), xycoords='data',
            xytext=(2, 0.33), textcoords='data',
            arrowprops=dict(arrowstyle="-", connectionstyle="arc3", linewidth=4))

draw_brace(ax, (0, 2), 0.39, '***')
draw_brace(ax, (0, 1), 0.34, '****')
draw_brace(ax, (1, 2), 0.29, '****')

sns.despine(fig=None, ax=None, top=True, right=True, left=False, bottom=False, offset=None, trim=False)

plt.text(0.5, 0.7, '')

# Spines
ax.spines['left'].set_linewidth(3)
ax.spines['bottom'].set_linewidth(3)

plt.savefig(fname='hopkins_of_all.tiff', dpi='figure', format='tiff',
        facecolor='white', edgecolor='auto'
       )

plt.show()


## Silhouette coefficient analysis

### For K-means clustering

In [ ]:
X = principalComponents

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score

import matplotlib.cm as cm

range_n_clusters = [2, 3, 4, 5, 6]

for n_clusters in range_n_clusters:
    # Create a subplot with 1 row and 2 columns
    fig, (ax1, ax2) = plt.subplots(1, 2)
    fig.set_size_inches(18, 7)

    # The 1st subplot is the silhouette plot
    # The silhouette coefficient can range from -1, 1
    ax1.set_xlim([-0.1, 1])
    # The (n_clusters+1)*10 is for inserting blank space between silhouette
    # plots of individual clusters, to demarcate them clearly.
    ax1.set_ylim([0, len(X) + (n_clusters + 1) * 10])

    # Initialize the clusterer with n_clusters value and a random generator
    # seed of 10 for reproducibility.
    clusterer = KMeans(n_clusters=n_clusters, n_init=1, random_state=10)
    cluster_labels = clusterer.fit_predict(X)

    # The silhouette_score gives the average value for all the samples.
    silhouette_avg = silhouette_score(X, cluster_labels)
    print(
        "For n_clusters =",
        n_clusters,
        "The average silhouette_score is :",
        silhouette_avg,
    )

    # Compute the silhouette scores for each sample
    sample_silhouette_values = silhouette_samples(X, cluster_labels)

    y_lower = 10
    for i in range(n_clusters):
        # Aggregate the silhouette scores for samples belonging to
        # cluster i, and sort them
        ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]

        ith_cluster_silhouette_values.sort()

        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i

        color = cm.nipy_spectral(float(i) / n_clusters)
        ax1.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            ith_cluster_silhouette_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7,
        )

        # Label the silhouette plots with their cluster numbers at the middle
        ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

        # Compute the new y_lower for next plot
        y_lower = y_upper + 10  # 10 for the 0 samples

    ax1.set_title("The silhouette plot for the various clusters.")
    ax1.set_xlabel("The silhouette coefficient values")
    ax1.set_ylabel("Cluster label")

    # The vertical line for average silhouette score of all the values
    ax1.axvline(x=silhouette_avg, color="red", linestyle="--")

    ax1.set_yticks([])  # Clear the yaxis labels / ticks
    ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

    # 2nd Plot showing the actual clusters formed
    colors = cm.nipy_spectral(cluster_labels.astype(float) / n_clusters)
    ax2.scatter(
        X[:, 0], X[:, 1], marker=".", s=300, lw=0, alpha=0.7, c=colors, edgecolor="k"
    )

    # Labeling the clusters
    centers = clusterer.cluster_centers_
    # Draw white circles at cluster centers
    ax2.scatter(
        centers[:, 0],
        centers[:, 1],
        marker="o",
        c="white",
        alpha=1,
        s=200,
        edgecolor="k",
    )

    for i, c in enumerate(centers):
        ax2.scatter(c[0], c[1], marker="$%d$" % i, alpha=1, s=50, edgecolor="k")

    ax2.set_title("The visualization of the clustered data.")
    ax2.set_xlabel("PC1")
    ax2.set_ylabel("PC2")

    plt.suptitle(
        "Silhouette analysis for KMeans clustering on sample data with n_clusters = %d"
        % n_clusters,
        fontsize=14,
        fontweight="bold",
    )

plt.show()